### Importing Modules

Install requirements.txt first!

In [1]:
import torch
print(torch.cuda.is_available())  # Should return True
print(torch.version.cuda)         # Should return 12.1
print(torch.backends.cudnn.version())  # Should print cuDNN version

True
12.1
90100


In [2]:
import os

# Completely disable GPU for TensorFlow
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"  # Ensure no GPU is used
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"  # Suppress TensorFlow logs

import tensorflow as tf

# Check if TensorFlow still detects any GPUs
print(tf.config.list_physical_devices("GPU"))  # Should return []

[]


In [3]:
import json
import chromadb
import gc
import time
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer, CrossEncoder
from langchain.text_splitter import RecursiveCharacterTextSplitter

### Check GPU and Loading JSON

In [4]:
### Check GPU
def check_gpu():
    """Verify if a GPU is present and return device type."""
    print("GPU load successful.")
    return "cuda" if torch.cuda.is_available() else "cpu"

### Load Texts
def load_json(file_path):
    """Load JSON data from file."""
    with open(file_path, 'r', encoding='utf-8') as f:
        print(f"JSON file loaded from {file_path}")
        return json.load(f)

### Creating ChromaDB

In [5]:
### Chunk Texts, Convert to Embedding, and Store into ChromaDB Vector Database
def create_chromadb(text_data, embedding_model, num_of_chunks, num_of_overlaps, chroma_path, batch_size=50):
    """Manually compute embeddings, save in ChromaDB, and store metadata."""

    start_time = time.time()  # Start time

    print("Initializing text splitter...")
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=num_of_chunks, chunk_overlap=num_of_overlaps)
    
    print("Initializing ChromaDB...")
    client = chromadb.PersistentClient(path=chroma_path)

    collection_name = f"{num_of_chunks}-{num_of_overlaps}_vectorstore"
    print(f"Everything can be found in: {collection_name}")

    collection = client.get_or_create_collection(name=collection_name)
    
    metadata_list = []
    
    for i in range(0, len(text_data), batch_size):
        batch_start_time = time.time()  # Start time for batch processing
        batch = text_data[i:i+batch_size]
        
        batch_embeddings = []
        batch_metadata = []

        for entry in batch:
            chunks = text_splitter.split_text(entry['full_text'])  
            for chunk in chunks:
                metadata = {
                    "acl_id": entry['acl_id'],
                    "author": entry['author'],
                    "title_on_paper": entry['title_on_paper'],
                    "year_on_paper": entry['year_on_paper'],
                    "citation_item": entry['citation_item'],
                    "chunked_text": chunk # Stores only the chunked text
                }
                batch_metadata.append(metadata)

                # Generate embedding
                embedding = embedding_model.encode(chunk, convert_to_numpy=True)
                batch_embeddings.append((embedding, metadata))
                
        # Add batch to ChromaDB
        if batch_embeddings:
            batch_number = i // batch_size + 1
            
            # Store embeddings and metadata in ChromaDB
            collection.add(
                embeddings=[emb[0].tolist() for emb in batch_embeddings],
                metadatas=[emb[1] for emb in batch_embeddings],
                ids=[str(i) for i in range(len(metadata_list), len(metadata_list) + len(batch_embeddings))]
            )
            
            # Append metadata to global list
            metadata_list.extend(batch_metadata)
            
            # Remove embedding from GPU memory to prevent OOM
            del batch_embeddings
            torch.cuda.empty_cache()
            
            batch_end_time = time.time()  # End time for batch processing
            batch_time_taken = batch_end_time - batch_start_time
            print(f"Added batch {batch_number} to ChromaDB. Time taken: {batch_time_taken:.2f} seconds.")
            print(f"Total of {batch_number * batch_size} documents out of {len(text_data)} documents.")
    
    total_time = time.time() - start_time  # Calculate total time
    print(f"Total documents stored in ChromaDB: {len(metadata_list)}")
    print(f"Total execution time: {total_time:.2f} seconds.")  # Print total execution time
    
    return collection

### Reranker Component

In [6]:
def retrieve_from_chroma(collection, query_embedding, number_of_docs=100):
    """
    Retrieve similar chunks from an already loaded ChromaDB collection.
    
    Args:
        collection: The already loaded ChromaDB collection.
        query_embedding: The query vector (NumPy array).
        number_of_docs: Number of similar chunks to return.
    
    Returns:
        List of retrieved chunks and their citation items.
    """
    # Convert query embedding to list format (required by ChromaDB)
    query_vector = query_embedding.tolist() if isinstance(query_embedding, np.ndarray) else query_embedding

    # Query ChromaDB
    results = collection.query(
        query_embeddings=[query_vector],  # Query must be a list
        n_results=number_of_docs
    )

    # Extract metadata list (metadata is inside a list)
    metadata_list = results["metadatas"][0]  # Get the first (and only) list

    # Extract citation items and chunked texts
    retrieved_citation_item = [entry.get("citation_item", "Unknown") for entry in metadata_list]
    retrieved_chunks = [entry.get("chunked_text", "") for entry in metadata_list]
    
    return retrieved_citation_item, retrieved_chunks

In [7]:
def rerank_results(query, retrieved_citation_items, retrieved_chunks, reranker_model, top_k):
    """Use cross-encoder to rerank retrieved texts, ensuring unique citation_items after reranking."""

    # Return early if there are no documents
    if not retrieved_chunks:
        return []
    
    # Extract text content efficiently
    citation_items = retrieved_citation_items

    # Lets get the scores
    results = reranker_model.rank(query, retrieved_chunks)

    scores = [item['score'] for item in results]

    # Use NumPy argsort for fast sorting (descending order)
    sorted_indices = np.argsort(scores)[::-1]

    # Keep only the highest-ranked document for each unique citation_item
    unique_citation_docs = [] # List to track unique citations

    for idx in sorted_indices:
        citation = citation_items[idx]

        if citation and citation not in unique_citation_docs:
            unique_citation_docs.append(citation) # Track this citation
            
        if len(unique_citation_docs) == top_k:  # Stop once we have enough unique results
            break

    return unique_citation_docs

In [8]:
def retrieve_and_match_citations(collection, embedding_model, reranker_model, query, citation_target, number_of_docs=100, top_k=10):
    """
    Retrieves and matches citations using the specified matching type.
    
    Args:
        collection: The ChromaDB collection.
        embedding_model: The embedding model used.
        reranker_model: The reranker model used.
        query (str): The input query.
        citation_target (str): The citation to match.
        number_of_docs: Number of documents to retrieve.
        top_k (int): Number of top retrieved documents.

    Returns:
        List of documents.
    """

    # Convert query to embedding
    query_embedding = embedding_model.encode(query, convert_to_numpy=True)

    # Retrieve and rerank documents
    citation_items, chunks = retrieve_from_chroma(collection, query_embedding, number_of_docs)
    unique_citation_items = rerank_results(query, citation_items, chunks, reranker_model, top_k)

    # Match citations
    matched_citations = []
    matched_position = []

    citation_target = citation_target.strip()
    citation_set = {citation_target}  # Use set for O(1) lookup

    for index, citation in enumerate(unique_citation_items):
        if citation.strip() in citation_set:
            matched_citations.append(citation)
            matched_position.append(index)
            break

    # Explicitly free memory from temporary variables if necessary
    del query_embedding, citation_items, chunks, unique_citation_items
    gc.collect()
    
    return matched_citations, matched_position

### Evaluation Component

In [9]:
def load_queries_from_csv(csv_path: str, just_context: bool):
    """Loads queries and citation targets from a CSV file using pandas."""

    df = pd.read_csv(csv_path, encoding="utf-8")

    if just_context:
        queries = df["masked_cit_context"]
    else:
        # Fix invalid escape sequences in citing_abstract
        df["citing_abstract"] = df["citing_abstract"].str.replace(r"\/", "/", regex=True)

        # Construct the full query using the new format:
        # masked_cit_context + " </s> " + citing_title + " </s> " + citing_abstract
        queries = (df["masked_cit_context"] + " </s> " + df["citing_title"] + " </s> " + df["citing_abstract"]).tolist()
        
    citation_targets = df["masked_token_target"].tolist()

    return queries, citation_targets

In [10]:
def compute_all_metrics_batches(csv_path, just_context, collection, embedding_model, reranker_model, number_of_docs, top_k, batch_size=100):
    """
    Evaluates citation retrieval over all queries using batching.
    Returns a dict with average metrics and the total number of queries processed.

    Args:
        csv_path (str): Path to the CSV file.
        just_context (bool): Whether to use just context.
        collection: The Chromadb containing the embeddings and metadata.
        embedding_model: The embedding model to use for encoding the queries.
        reranker_model: The reranker model used after initial retrieval.
        number_of_docs (int): Number of documents to retrieve in the first pass.
        top_k (int): Number of top retrieved documents to consider for matching.
        batch_size (int): Number of queries to process per batch.
    
    Returns:
        dict: {
            "Recall": float,
            "MRR": float,
            "NDCG": float,
            "count": int
        }
    """
    all_queries, all_targets = load_queries_from_csv(csv_path, just_context)
    all_queries = all_queries[:100]
    all_targets = all_targets[:100]
    print(f"Total Number of Queries to be Evaluated: {len(all_queries)}")

    recall_scores = []
    mrr_scores = []
    ndcg_scores = []
    
    total_start_time = time.time()

    # Process queries in batches
    for batch_start in range(0, len(all_queries), batch_size):
        batch_queries = all_queries[batch_start:batch_start + batch_size]
        batch_targets = all_targets[batch_start:batch_start + batch_size]

        for query, citation_target in zip(batch_queries, batch_targets):
            matched_citations, matched_position = retrieve_and_match_citations(
                collection,
                embedding_model, 
                reranker_model, 
                query, 
                citation_target,
                number_of_docs, 
                top_k
            )

            # Compute metrics for this query
            if matched_position:
                pos = matched_position[0]
                recall_scores.append(1)
                mrr_scores.append(1 / (pos + 1))
                ndcg_scores.append(1 / np.log2(pos + 2))
            else:
                recall_scores.append(0)
                mrr_scores.append(0)
                ndcg_scores.append(0)

        # Optionally, print batch statistics
        batch_elapsed = time.time() - total_start_time
        print(f"Processed batch ending at query {min(batch_start + batch_size, len(all_queries))} out of {len(all_queries)} in {batch_elapsed:.2f} seconds")

        # Compute and print running metrics
        running_avg_recall = np.mean(recall_scores)
        running_avg_mrr = np.mean(mrr_scores)
        running_avg_ndcg = np.mean(ndcg_scores)
        
        print(f"Running Metrics: Average Recall: {running_avg_recall:.4f} | Average MRR: {running_avg_mrr:.4f} | Average NDCG: {running_avg_ndcg:.4f}")

    # Aggregate metrics over all queries
    count = len(all_queries)
    overall_metrics = {
        "Recall": np.mean(recall_scores) if count > 0 else 0,
        "MRR": np.mean(mrr_scores) if count > 0 else 0,
        "NDCG": np.mean(ndcg_scores) if count > 0 else 0,
        "count": count
    }
    
    print("Overall Metrics:")
    print(overall_metrics)
    
    return overall_metrics

### To Run

In [11]:
device = check_gpu()
print(f"GPU: {device}")

# Load data
full_text_data = load_json("/home/ubuntu/Infly-Chroma-Bread Pipeline/cleaned_global_train_eval_context_list.json")

full_text_data = full_text_data[:100]

# Define embedding model
embedding_model_name = "infly/inf-retriever-v1-1.5b"
embedding_model = SentenceTransformer(embedding_model_name, device=device)
print(f"Embedding model loaded: {embedding_model_name}")

# Chunking parameters
num_of_chunks = 1024
num_of_overlaps = 256

print(f"Number of Chunks: {num_of_chunks}")
print(f"Number of Overlaps: {num_of_overlaps}")

# Path to the Chroma DB
chroma_path = f"./ChromaDB Storage Experiments"


# Create ChromaDB vector store
collection = create_chromadb(
    text_data=full_text_data,
    embedding_model=embedding_model,
    num_of_chunks=num_of_chunks,
    num_of_overlaps=num_of_overlaps,
    chroma_path=chroma_path,
    batch_size=50
)


# Load the existing ChromaDB database
client = chromadb.PersistentClient(path=chroma_path)

collection_name = f"{num_of_chunks}-{num_of_overlaps}_vectorstore"

# Load the collection
loaded_collection = client.get_collection(name=collection_name)

# Check stored data
print("Collection loaded:", collection_name)
print(loaded_collection.count())

# Define reranker model
reranker_model_name = "mixedbread-ai/mxbai-rerank-large-v1"
reranker_model = CrossEncoder(reranker_model_name, device=device)
print(f"Reranker model loaded: {reranker_model_name}")

# Number of documents to retrieve
number_of_docs = 100

# Varying top k documents to get from rerank
top_k = 10
 
# Evaluate Reranker
eval_csv_path = "/home/ubuntu/Infly-Chroma-Bread Pipeline/final_cleaned_acl_global_context_dataset_eval.csv" # Update with your actual file path

# Just context or not
just_context = True

# Batch size
batch_size = 50

print("Other parameters loaded successfully!")

print(f"Evaluating for {just_context}_Query Only-{num_of_chunks}-{num_of_overlaps}-{embedding_model_name}-{reranker_model_name}-{number_of_docs}-top_{top_k}")

overall_metrics = compute_all_metrics_batches(
    csv_path=eval_csv_path,  # Path to your CSV file
    just_context=just_context, # Just context or not
    collection=loaded_collection, # ChromaDB collection
    embedding_model=embedding_model,  # Replace with your embedding model
    reranker_model=reranker_model,   # Replace with your reranker model
    number_of_docs=number_of_docs, # Number of docs to retrieve
    top_k=top_k, # Top k documents to retain
    batch_size=batch_size # How many to evaluate in a batch
)

print(f"Done Evaluating for {just_context}_Query Only-{num_of_chunks}-{num_of_overlaps}-{embedding_model_name}-{reranker_model_name}-{number_of_docs}-top_{top_k}")

GPU load successful.
GPU: cuda
JSON file loaded from /home/ubuntu/Infly-Chroma-Bread Pipeline/cleaned_global_train_eval_context_list.json


Embedding model loaded: infly/inf-retriever-v1-1.5b
Number of Chunks: 1024
Number of Overlaps: 256
Initializing text splitter...
Initializing ChromaDB...
Everything can be found in: 1024-256_vectorstore
Added batch 1 to ChromaDB. Time taken: 97.50 seconds.
Total of 50 documents out of 100 documents.
Added batch 2 to ChromaDB. Time taken: 101.25 seconds.
Total of 100 documents out of 100 documents.
Total documents stored in ChromaDB: 3501
Total execution time: 199.32 seconds.
Collection loaded: 1024-256_vectorstore
3501
Reranker model loaded: mixedbread-ai/mxbai-rerank-large-v1
Other parameters loaded successfully!
Evaluating for True_Query Only-1024-256-infly/inf-retriever-v1-1.5b-mixedbread-ai/mxbai-rerank-large-v1-100-top_10
Total Number of Queries to be Evaluated: 100
Processed batch ending at query 50 out of 100 in 153.92 seconds
Running Metrics: Average Recall: 0.8600 | Average MRR: 0.5819 | Average NDCG: 0.6471
Processed batch ending at query 100 out of 100 in 306.67 seconds
Runn

### Check Retrieved Score

In [13]:
device = check_gpu()
print(f"GPU: {device}")

# Define embedding model
embedding_model_name = "infly/inf-retriever-v1-1.5b"
embedding_model = SentenceTransformer(embedding_model_name, device=device)
print(f"Embedding model loaded: {embedding_model_name}")

GPU load successful.
GPU: cuda


Embedding model loaded: infly/inf-retriever-v1-1.5b
[-0.02201182  0.0593452   0.03764375 ... -0.03676639  0.0034566
  0.00813164]


In [16]:
query = "What is FrameNet Project?"

# Convert query to embedding
query_embedding = embedding_model.encode(query, convert_to_numpy=True)

print(query_embedding)

# Chunking parameters
num_of_chunks = 1024
num_of_overlaps = 256

print(f"Number of Chunks: {num_of_chunks}")
print(f"Number of Overlaps: {num_of_overlaps}")

# Path to the Chroma DB
chroma_path = f"./ChromaDB Storage Experiments"

# Load the existing ChromaDB database
client = chromadb.PersistentClient(path=chroma_path)

collection_name = f"{num_of_chunks}-{num_of_overlaps}_vectorstore"

# Load the collection
loaded_collection = client.get_collection(name=collection_name)

# Query ChromaDB
results = loaded_collection.query(
    query_embeddings=query_embedding,  # Query must be a list
    n_results=10
)

results

# Lower score is better

[ 0.01207809  0.0256633   0.03005284 ... -0.02098976  0.01425658
  0.00909485]
Number of Chunks: 1024
Number of Overlaps: 256


{'ids': [['1', '11', '3', '12', '0', '5', '2', '6', '10', '4']],
 'embeddings': None,
 'documents': [[None, None, None, None, None, None, None, None, None, None]],
 'uris': None,
 'data': None,
 'metadatas': [[{'acl_id': 'P98-1013',
    'author': 'Baker, Collin F. and Fillmore, Charles J. and Lowe, John B.',
    'chunked_text': 'the observed linkings between "frame elements" and their syntactic realizations (e.g. grammatical function, phrase type, and other syntactic traits). This report will present the project\'s goals and workflow, and information about the computational tools that have been adapted or created in-house for this work. Introduction The Berkeley FrameNet project 1 is producing frame-semantic descriptions of several thousand English lexical items and backing up these descriptions with semantically annotated attestations from contemporary English corpora 2. 1The project is based at the International Computer Science Institute (1947 Center Street, Berkeley, CA) . A fuller

### Checking Retrieved Contexts only

In [13]:
device = check_gpu()
print(f"GPU: {device}")

# Load data
full_text_data = load_json("/home/ubuntu/Infly-Chroma-Bread Pipeline/cleaned_global_train_eval_context_list.json")

full_text_data = full_text_data[:100]

# Define embedding model
embedding_model_name = "infly/inf-retriever-v1-1.5b"
embedding_model = SentenceTransformer(embedding_model_name, device=device)
print(f"Embedding model loaded: {embedding_model_name}")

# Chunking parameters
num_of_chunks = 1024
num_of_overlaps = 256

print(f"Number of Chunks: {num_of_chunks}")
print(f"Number of Overlaps: {num_of_overlaps}")

# Path to the Chroma DB
chroma_path = f"./ChromaDB Storage Experiments"

# Load the existing ChromaDB database
client = chromadb.PersistentClient(path=chroma_path)

collection_name = f"{num_of_chunks}-{num_of_overlaps}_vectorstore"

# Load the collection
loaded_collection = client.get_collection(name=collection_name)

# Check stored data
print("Collection loaded:", collection_name)
print(loaded_collection.count())

# Number of documents to retrieve
number_of_docs = 100

# Varying top k documents to get from rerank
top_k = 10
 
# Evaluate Reranker
eval_csv_path = "/home/ubuntu/Infly-Chroma-Bread Pipeline/final_cleaned_acl_global_context_dataset_eval.csv" # Update with your actual file path

# Just context or not
just_context = True

queries, citation_targets = load_queries_from_csv(csv_path=eval_csv_path, just_context=just_context)

queries = queries[:100]

print(len(queries))

retrieved_chunks = []
retrieved_citations = []

for i, query in enumerate(queries):

    # Print a status update every 100 queries
    if i % 100 == 0:
        print(f"Processing is ongoing: {i} out of {len(queries)} queries processed.")

    # Convert query to embedding
    query_embedding = embedding_model.encode(query, convert_to_numpy=True)
    citation_items, chunks = retrieve_from_chroma(loaded_collection, query_embedding, number_of_docs=100)

    # Appends the retrieved items to their respective list
    retrieved_chunks.append(chunks)
    retrieved_citations.append(citation_items)

chunks_file_name = f"{num_of_chunks}-{num_of_overlaps}-retrieved_chunks.json"
with open(chunks_file_name, "w") as file:
    json.dump(retrieved_chunks, file, indent=4)

print("Retrieved chunks saved as a .json file.")

citations_file_name = f"{num_of_chunks}-{num_of_overlaps}-retrieved_citations.json"
with open(citations_file_name, "w") as file:
    json.dump(retrieved_citations, file, indent=4)

print("Retrieved citations saved as a .json file.")

GPU load successful.
GPU: cuda
JSON file loaded from /home/ubuntu/Infly-Chroma-Bread Pipeline/cleaned_global_train_eval_context_list.json
Embedding model loaded: infly/inf-retriever-v1-1.5b
Number of Chunks: 1024
Number of Overlaps: 256
Collection loaded: 1024-256_vectorstore
3501
100
Processing is ongoing: 0 out of 100 queries processed.
Retrieved chunks saved as a .json file.
Retrieved citations saved as a .json file.


### Checking the Rerank and Eval

In [12]:
def rerank_results(query, retrieved_citation_items, retrieved_chunks, reranker_model, top_k):
    """Use cross-encoder to rerank retrieved texts, ensuring unique citation_items after reranking."""

    # Return early if there are no documents
    if not retrieved_chunks:
        return []
    
    # Extract text content efficiently
    citation_items = retrieved_citation_items

    # Lets get the scores
    results = reranker_model.rank(query, retrieved_chunks)

    scores = [item['score'] for item in results]

    # Use NumPy argsort for fast sorting (descending order)
    sorted_indices = np.argsort(scores)[::-1]

    # Keep only the highest-ranked document for each unique citation_item
    unique_citation_docs = [] # List to track unique citations

    for idx in sorted_indices:
        citation = citation_items[idx]

        if citation and citation not in unique_citation_docs:
            unique_citation_docs.append(citation) # Track this citation
            
        if len(unique_citation_docs) == top_k:  # Stop once we have enough unique results
            break

    return unique_citation_docs


In [13]:
def match_citations(query, retrieved_citation_items, retrieved_chunks, reranker_model,  rerank, citation_target, top_k=10):
    """
    Matches citations.
    
    Args:
        query (str): The input query.
        retrieved_citation_items: list of retrieved citations
        retrieved_chunks: list of retrieved chunks
        reranker_model: the reranker model to be used
        rerank: True if you want to rerank, False otherwise
        citation_target (str): The citation to match.
        top_k (int): Number of top retrieved documents.

    Returns:
        List of documents.
    """

    if rerank:
        unique_citation_items = rerank_results(query, retrieved_citation_items, retrieved_chunks, reranker_model, top_k)
    else:

        # Keep only the highest-ranked document for each unique citation_item
        unique_citation_items = [] # List to track unique citations

        for idx in retrieved_citation_items:
            citation = retrieved_citation_items[idx]

            if citation and citation not in unique_citation_items:
                unique_citation_items.append(citation) # Track this citation
                
            if len(unique_citation_items) == top_k:  # Stop once we have enough unique results
                break

    # Match citations
    matched_citations = []
    matched_position = []

    citation_target = citation_target.strip()
    citation_set = {citation_target}  # Use set for O(1) lookup

    for index, citation in enumerate(unique_citation_items):
        if citation.strip() in citation_set:
            matched_citations.append(citation)
            matched_position.append(index)
            break

    return matched_citations, matched_position

In [14]:
def compute_all_metrics_batches(csv_path, just_context, chunks_file_path, citations_file_path, rerank, reranker_model, top_k, batch_size=100):
    """
    Evaluates citation retrieval over all queries using batching.
    Returns a dict with average metrics and the total number of queries processed.

    Args:
        csv_path (str): Path to the CSV file.
        just_context (bool): Whether to use just context.
        collection: The Chromadb containing the embeddings and metadata.
        embedding_model: The embedding model to use for encoding the queries.
        reranker_model: The reranker model used after initial retrieval.
        number_of_docs (int): Number of documents to retrieve in the first pass.
        top_k (int): Number of top retrieved documents to consider for matching.
        batch_size (int): Number of queries to process per batch.
    
    Returns:
        dict: {
            "Recall": float,
            "MRR": float,
            "NDCG": float,
            "count": int
        }
    """
    # Load queries and target citations
    all_queries, all_targets = load_queries_from_csv(csv_path, just_context)
    print(f"Total Number of Queries to be Evaluated: {len(all_queries)}")

    # Load the chunks and citations
    retrieved_chunks = load_json(chunks_file_path)
    retrieved_citations = load_json(citations_file_path)

    recall_scores = []
    mrr_scores = []
    ndcg_scores = []
    
    total_start_time = time.time()

    # Process queries in batches
    for batch_start in range(0, len(all_queries), batch_size):

        batch_queries = all_queries[batch_start:batch_start + batch_size]
        batch_targets = all_targets[batch_start:batch_start + batch_size]
        batch_chunks = retrieved_chunks[batch_start:batch_start + batch_size]
        batch_citations = retrieved_citations[batch_start:batch_start + batch_size]

        for query, citation_target, chunk, citation in zip(batch_queries, batch_targets, batch_chunks, batch_citations):
            matched_citations, matched_position = match_citations(
                query=query,
                retrieved_citation_items=citation, 
                retrieved_chunks=chunk,
                reranker_model=reranker_model,
                rerank=rerank,
                citation_target=citation_target,
                top_k=top_k
            )

            # Compute metrics for this query
            if matched_position:
                pos = matched_position[0]
                recall_scores.append(1)
                mrr_scores.append(1 / (pos + 1))
                ndcg_scores.append(1 / np.log2(pos + 2))
            else:
                recall_scores.append(0)
                mrr_scores.append(0)
                ndcg_scores.append(0)

        # Optionally, print batch statistics
        batch_elapsed = time.time() - total_start_time
        print(f"Processed batch ending at query {min(batch_start + batch_size, len(all_queries))} out of {len(all_queries)} in {batch_elapsed:.2f} seconds")

        # Compute and print running metrics
        running_avg_recall = np.mean(recall_scores)
        running_avg_mrr = np.mean(mrr_scores)
        running_avg_ndcg = np.mean(ndcg_scores)
        
        print(f"Running Metrics: Average Recall: {running_avg_recall:.4f} | Average MRR: {running_avg_mrr:.4f} | Average NDCG: {running_avg_ndcg:.4f}")

    # Aggregate metrics over all queries
    count = len(all_queries)
    overall_metrics = {
        "Recall": np.mean(recall_scores) if count > 0 else 0,
        "MRR": np.mean(mrr_scores) if count > 0 else 0,
        "NDCG": np.mean(ndcg_scores) if count > 0 else 0,
        "count": count
    }
    
    print("Overall Metrics:")
    print(overall_metrics)
    
    return overall_metrics


In [15]:
### To Run

device = check_gpu()
print(f"GPU: {device}")

# Chunking parameters
num_of_chunks = 1024
num_of_overlaps = 256

print(f"Number of Chunks: {num_of_chunks}")
print(f"Number of Overlaps: {num_of_overlaps}")

chunks_file_path = f"/home/ubuntu/Infly-Chroma-Bread Pipeline/Retrieved Chunks/{num_of_chunks}-{num_of_overlaps}-retrieved_chunks.json"
citations_file_path = f"/home/ubuntu/Infly-Chroma-Bread Pipeline/Retrieved Citations/{num_of_chunks}-{num_of_overlaps}-retrieved_citations.json"
print("Chunks and Citations file path loaded.")

# Define reranker model
reranker_model_name = "mixedbread-ai/mxbai-rerank-large-v1"
reranker_model = CrossEncoder(reranker_model_name, device=device)
print(f"Reranker model loaded: {reranker_model_name}")

# Number of documents to retrieve
number_of_docs = 100

# Varying top k documents to get from rerank
top_k = 10
 
# Evaluate Reranker
eval_csv_path = "/home/ubuntu/Infly-Chroma-Bread Pipeline/final_cleaned_acl_global_context_dataset_eval.csv" # Update with your actual file path

# Just context or not
just_context = True

# Will rerank or not
rerank = True

# Batch size
batch_size = 50

print("Other parameters loaded successfully!")

print(f"Evaluating for {just_context}_Query_Only-{num_of_chunks}-{num_of_overlaps}-Infly-ChromaDB-Bread-{number_of_docs}-top_{top_k}")

overall_metrics = compute_all_metrics_batches(
    csv_path=eval_csv_path,  # Path to your CSV file
    just_context=just_context, # Just context or not
    chunks_file_path=chunks_file_path, # Chunks file path
    citations_file_path=citations_file_path, # Citations file path
    rerank=rerank, # True if will rerank and False if not
    reranker_model=reranker_model,   # Replace with your reranker model
    top_k=top_k, # Top k documents to retain
    batch_size=batch_size # How many to evaluate in a batch
)

print(f"Done Evaluating for {just_context}_Query Only-{num_of_chunks}-{num_of_overlaps}-Infly-ChromaDB-Bread-{number_of_docs}-top_{top_k}")


GPU load successful.
GPU: cuda
Number of Chunks: 1024
Number of Overlaps: 256
Chunks and Citations file path loaded.


Reranker model loaded: mixedbread-ai/mxbai-rerank-large-v1
Other parameters loaded successfully!
Evaluating for True_Query_Only-1024-256-Infly-ChromaDB-Bread-100-top_10
Total Number of Queries to be Evaluated: 12450
JSON file loaded from /home/ubuntu/Infly-Chroma-Bread Pipeline/Retrieved Chunks/1024-256-retrieved_chunks.json
JSON file loaded from /home/ubuntu/Infly-Chroma-Bread Pipeline/Retrieved Citations/1024-256-retrieved_citations.json
Processed batch ending at query 50 out of 12450 in 139.29 seconds
Running Metrics: Average Recall: 0.8600 | Average MRR: 0.5819 | Average NDCG: 0.6471


KeyboardInterrupt: 

### New Reranker

In [6]:
from sentence_transformers import CrossEncoder

# Load the model, here we use our base sized model
model = CrossEncoder("mixedbread-ai/mxbai-rerank-large-v1")


# Example query and documents
query = "Who wrote 'To Kill a Mockingbird'?"
documents = [
    "'To Kill a Mockingbird' is a novel by Harper Lee published in 1960. It was immediately successful, winning the Pulitzer Prize, and has become a classic of modern American literature.",
    "The novel 'Moby-Dick' was written by Herman Melville and first published in 1851. It is considered a masterpiece of American literature and deals with complex themes of obsession, revenge, and the conflict between good and evil.",
    "Harper Lee, an American novelist widely known for her novel 'To Kill a Mockingbird', was born in 1926 in Monroeville, Alabama. She received the Pulitzer Prize for Fiction in 1961.",
    "Jane Austen was an English novelist known primarily for her six major novels, which interpret, critique and comment upon the British landed gentry at the end of the 18th century.",
    "The 'Harry Potter' series, which consists of seven fantasy novels written by British author J.K. Rowling, is among the most popular and critically acclaimed books of the modern era.",
    "'The Great Gatsby', a novel written by American author F. Scott Fitzgerald, was published in 1925. The story is set in the Jazz Age and follows the life of millionaire Jay Gatsby and his pursuit of Daisy Buchanan."
]

# Lets get the scores
results = model.rank(query, documents, return_documents=True, top_k=3)

print(results)

for i in results:
    print(i['score'])
    print(i['corpus_id'])

[{'corpus_id': 0, 'score': 0.99801946, 'text': "'To Kill a Mockingbird' is a novel by Harper Lee published in 1960. It was immediately successful, winning the Pulitzer Prize, and has become a classic of modern American literature."}, {'corpus_id': 2, 'score': 0.9969399, 'text': "Harper Lee, an American novelist widely known for her novel 'To Kill a Mockingbird', was born in 1926 in Monroeville, Alabama. She received the Pulitzer Prize for Fiction in 1961."}, {'corpus_id': 5, 'score': 0.02947863, 'text': "'The Great Gatsby', a novel written by American author F. Scott Fitzgerald, was published in 1925. The story is set in the Jazz Age and follows the life of millionaire Jay Gatsby and his pursuit of Daisy Buchanan."}]
0.99801946
0
0.9969399
2
0.02947863
5


In [3]:
from mxbai_rerank import MxbaiRerankV2

model = MxbaiRerankV2("mixedbread-ai/mxbai-rerank-large-v2")

query = "Who wrote 'To Kill a Mockingbird'?"
documents = [
    "'To Kill a Mockingbird' is a novel by Harper Lee published in 1960. It was immediately successful, winning the Pulitzer Prize, and has become a classic of modern American literature.",
    "The novel 'Moby-Dick' was written by Herman Melville and first published in 1851. It is considered a masterpiece of American literature and deals with complex themes of obsession, revenge, and the conflict between good and evil.",
    "Harper Lee, an American novelist widely known for her novel 'To Kill a Mockingbird', was born in 1926 in Monroeville, Alabama. She received the Pulitzer Prize for Fiction in 1961.",
    "Jane Austen was an English novelist known primarily for her six major novels, which interpret, critique and comment upon the British landed gentry at the end of the 18th century.",
    "The 'Harry Potter' series, which consists of seven fantasy novels written by British author J.K. Rowling, is among the most popular and critically acclaimed books of the modern era.",
    "'The Great Gatsby', a novel written by American author F. Scott Fitzgerald, was published in 1925. The story is set in the Jazz Age and follows the life of millionaire Jay Gatsby and his pursuit of Daisy Buchanan."
]

# Lets get the scores
results = model.rank(query, documents)

scores = [result.score for result in results]
print(scores)  # This would print: [11.5, 11.0, 2.625, 0.6875, 0.625, -0.3125]

indexes = [result.index for result in results]
print(indexes)


You're using a Qwen2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


[11.5, 11.0, 2.625, 0.6875, 0.625, -0.3125]
[0, 2, 4, 3, 5, 1]


/opt/conda/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:2708: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(


: 

### FlagLLM Reranker

In [1]:
from FlagEmbedding import LayerWiseFlagLLMReranker
import numpy as np

reranker = LayerWiseFlagLLMReranker('BAAI/bge-reranker-v2-minicpm-layerwise', use_fp16=True, device="cuda") # Setting use_fp16 to True speeds up computation with a slight performance degradation
# reranker = LayerWiseFlagLLMReranker('BAAI/bge-reranker-v2-minicpm-layerwise', use_bf16=True) # You can also set use_bf16=True to speed up computation with a slight performance degradation

score = reranker.compute_score(['query', 'passage'], cutoff_layers=[28]) # Adjusting 'cutoff_layers' to pick which layers are used for computing the score.
print(score)

scores = reranker.compute_score([['what is panda?', 'hi'], ['what is panda?', 'The giant panda (Ailuropoda melanoleuca), sometimes called a panda bear or simply panda, is a bear species endemic to China.']], cutoff_layers=[28])
print(scores)

sorted_indices = np.argsort(scores)[::-1]
print(sorted_indices)

2025-03-24 03:57:29.416300: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
LayerWiseMiniCPMForCausalLM has generative capabilities, as `prepare_inputs_for_generation` is explicitly overwritten. However, it doesn't directly inherit from `GenerationMixin`. From 👉v4.50👈 onwards, `PreTrainedModel` will NOT inherit from `GenerationMixin`, and this model will lose the ability to call `generate` and other related functions.
  - If you're using `trust_remote_code=True`, you can get rid of this warning by loading the model with an auto class. See https://huggingface.co/docs/transformers/en/model_doc/auto#auto-classes
  - If you are the owner of the model architecture code, please modify your model class such that it inherits from `GenerationMixin` (after `PreTra

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

You're using a LlamaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
100%|██████████| 1/1 [00:00<00:00, 22.55it/s]


[-7.06640625]


100%|██████████| 1/1 [00:00<00:00, 35.53it/s]

[-9.9453125, 1.7822265625]
[1 0]


In [15]:
bruh = load_json("/home/ubuntu/RAG Pipeline/Retrieved Chunks/100 Docs/1024-256-retrieved_chunks.json")

for i in range(2):
    for j in range(3):
        print(bruh[i][j])
        print("\n")

JSON file loaded from /home/ubuntu/RAG Pipeline/Retrieved Chunks/100 Docs/1024-256-retrieved_chunks.json
in argument identification on the FrameNet test set, leading to a 1% F 1 improvement on the full frame-semantic parsing task. Our code and models are available at http://www.ark.cs.cmu.edu/ SEMAFOR/. FrameNet FrameNet represents events, scenarios, and relationships with an inventory of frames (such as SHOPPING and SCARCITY). Each frame is associated with a set of roles (or frame elements) called to mind in order to understand the scenario, and lexical predicates (verbs, nouns, adjectives, and adverbs) capable of evoking the scenario. For example, the BODY_MOVEMENT frame has Agent and Body_part as its core roles, and lexical entries including verbs such as bend, blink, crane, and curtsy, plus the noun use of curtsy. In FrameNet 1.5, there are over 1,000 frames and 12,000 lexical predicates. Hierarchy The FrameNet lexicon is organized as a network, with several kinds of frame-to-frame